# Stage 1 — GPU benchmark + probing sweep

Runs `scripts/run_logit_benchmarks.py` and `scripts/run_probing.py` on a Colab GPU. Results are written to Google Drive so they persist across disconnects; the scripts auto-resume from per-`(model, benchmark)` JSON checkpoints if the runtime is killed.

**Setup (once):** Runtime → Change runtime type → GPU (A100 recommended). Sidebar key icon → add Colab Secrets `HF_TOKEN` and `WANDB_API_KEY` with notebook access enabled.

In [ ]:
GITHUB_REPO = 'https://github.com/korneli777/llm-bias-eval.git'
FAMILY = 'mistral'  # mistral | llama | qwen | gemma
DRIVE_DIR = '/content/drive/MyDrive/llm-bias-eval'

In [ ]:
import os, subprocess, shutil
from google.colab import drive, userdata

REPO_DIR = '/content/llm-bias-eval'

drive.mount('/content/drive')
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'pull', '--ff-only'], check=True)

results_link = os.path.join(REPO_DIR, 'results')
if os.path.lexists(results_link):
    (os.unlink if os.path.islink(results_link) else shutil.rmtree)(results_link)
os.symlink(f'{DRIVE_DIR}/results', results_link)

subprocess.run(['pip', 'install', '-q', '-e', REPO_DIR], check=True)

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['WANDB_PROJECT'] = 'llm-bias-eval'

In [ ]:
!python scripts/run_logit_benchmarks.py --family {FAMILY}

In [ ]:
!python scripts/run_probing.py --family {FAMILY}